In [201]:
TESTSTR_TL2POJ = ''''''
    # 白話字 -> 台羅
TESTSTR_POJ2TL = ''''''
    # 數字標 -> 符號標
TESTSTR_NO2DIAC = '''Jip8-tang1 i2-au7, m7-na7 thinn1-khi3 tsuan2-ling2, liu5-hing5-sing3 kam2-moo7 e5 penn7-tok8 ma7 tue3-leh4 khai1-si2 uah8-thiau3, tso7-sing5 put4-tsi2-a2 tse7 hak8-sing1 gin2-a2 kap4 tua7-lang5 tioh8-penn7.'''
    # 符號標 -> 數字標
TESTSTR_DIAC2NO = '''Ji̍p-tang í-āu, m̄-nā thinn-khì tsuán-líng, liû-hîng-sìng kám-mōo ê pēnn-to̍k mā tuè-leh khai-sí ua̍h-thiàu, tsō-sîng put-tsí-á tsē ha̍k-sing gín-á kap tuā-lâng tio̍h-pēnn.'''

In [178]:
class PingImSyllable:
    # 定義合法的音節組件
    INITIALS = {'p', 'ph', 'm', 'b', 't', 'th', 'n', 'l', 'k', 'kh', 'ng', 'g', 'ts', 'tsh', 's', 'j', 'h'}
    VOWELS = {'a', 'i', 'u', 'e', 'oo', 'o'}
    FINALS = {'n', 'ng', 'p', 't', 'k', 'h', 'm', 'nn'}
    VOWELIZED_CONSONANTS = {'ng','m'}
    CHECKED_TONES = ('p','t','k','h')
    
    # 母音優先順序
    VOWEL_PRIORITY = {'a':3,
                      'oo':2, 
                      'e':1, 
                      'o':1, 
                      'i':0, 
                      'u':0}
    
    TONE_MARKS = {
        'a': {'2': 'á', '3': 'à', '5': 'â', '6': 'ǎ', '7': 'ā', '8': 'a̍'},
        'i': {'2': 'í', '3': 'ì', '5': 'î', '6': 'ǐ', '7': 'ī', '8': 'i̍'},
        'u': {'2': 'ú', '3': 'ù', '5': 'û', '6': 'ǔ', '7': 'ū', '8': 'u̍'},
        'e': {'2': 'é', '3': 'è', '5': 'ê', '6': 'ě', '7': 'ē', '8': 'e̍'},
        'oo': {'2': 'óo', '3': 'òo', '5': 'ôo', '6': 'ǒo', '7': 'ōo', '8': 'o̍o'},
        'o': {'2': 'ó', '3': 'ò', '5': 'ô', '6': 'ǒ', '7': 'ō', '8': 'o̍'},
        'ng': {'2': 'ńg', '3': 'ǹg', '5': '̂n̂g', '6': 'ňg', '7': 'n̄g', '8': 'n̍g'},
        'm': {'2': 'ḿ', '3': 'm̀', '5': 'm̂', '6': 'm̌', '7': 'm̄', '8': 'm̍'}
    }

    def __init__(self, syllable: str, tone: str):
        """
        初始化拼音音節處理器
        :param syllable: 拼音音節（不含聲調）
        :param tone: 聲調（1-4）
        """
        self.original = syllable
        self.tone = tone
        self.initial = ''
        self.vowels = []
        self.final = ''
        self.main_vowel = ''
        
    def parse(self):
        """解析音節結構"""
        # 檢查是否為空
        if not self.original:
            raise ValueError("音節不能為空")
            
        # 找出聲母
        current = self.original
        for initial in sorted(self.INITIALS, key=len, reverse=True):
            if current.startswith(initial) and len(current) > 1:
                self.initial = initial
                current = current[len(initial):]
                break
                
        # 找出韻母中的所有母音
        i = 0
        while i < len(current):
            if current[i] in self.VOWELS:
                self.vowels.append(current[i])
            i += 1
            
        # 檢查母音數量
        if not 1 <= len(self.vowels) <= 3:
            for cons in self.VOWELIZED_CONSONANTS:
                if current.endswith(cons):
                    self.vowels.append(cons)
                    break

        if not 1 <= len(self.vowels) <= 3:
            # print(current)
            raise ValueError(f"母音數量必須在1-3個之間，目前有{len(self.vowels)}個")
            
        # 尋找主要母音
        self.main_vowel = self._find_main_vowel()
        
        # 找出韻尾
        for final in sorted(self.FINALS, key=len, reverse=True):
            if current.endswith(final):
                self.final = final
                current = current[:-len(final)]
                break
        return self
        
    def _find_main_vowel(self) -> str:
        """根據優先順序找出主要母音"""
        if not self.vowels:
            raise ValueError("沒有找到母音")
            
        # 根據優先順序表找出優先級最高的母音
        highest_priority = -1
        main_vowel = ''
        
        if self.vowels[0] in self.VOWELIZED_CONSONANTS:
            main_vowel = self.vowels[0]
            return main_vowel
        
        for vowel in self.vowels:
            priority = self.VOWEL_PRIORITY[vowel]
            if priority >= highest_priority: #後來者會替換前者
                highest_priority = priority
                main_vowel = vowel
                
        return main_vowel
        
    def add_tone_mark(self) -> str:

        if not self.main_vowel or not self.tone:
            raise ValueError("缺少主要母音或聲調信息")
            
        if self.tone not in {'1', '2', '3','4', '5', '6', '7', '8'}:
            raise ValueError("聲調必須是1-8之間的數字")
            
        # 獲取帶聲調的母音
        if self.tone in {'1','4'}:
            toned_vowel = self.main_vowel
        else:
            toned_vowel = self.TONE_MARKS[self.main_vowel][self.tone]
        
        # 替換原始字符串中的主要母音
        result = self.original.replace(self.main_vowel,toned_vowel)

        return result

In [146]:
import re
import unicodedata

class TaigiModes:
    # 台羅 -> 白話
    MODE_TL2POJ = 0
    # 白話字 -> 台羅
    MODE_POJ2TL = 1
    # 數字標 -> 符號標
    MODE_NO2DIAC = 2
    # 符號標 -> 數字標
    MODE_DIAC2NO = 3

tone_marks:dict = {'0301':'2','0300':'3','0302':'5','030c':'6','0304':'7','030d':'8','030b':'9',
                   '2':'0301','3':'0300','5':'0302','6':'030c','7':'0304','8':'030d','9':'030b'}

def process_pinyin(syllable: str, tone: str) -> str:
    """
    處理單個拼音音節
    :param syllable: 拼音音節（不含聲調）
    :param tone: 聲調(1-4)
    :return: 處理後的拼音（含變音符號）
    """
    processor = PingImSyllable(syllable, tone)
    processor.parse()
    return processor.add_tone_mark()

def diacritic_removal(syllable:str) -> str:
    normalized = unicodedata.normalize('NFD',syllable)
    tone=''
    for idx, letter in enumerate(normalized):
        if unicodedata.category(letter) == 'Mn' and letter != '\u0358' :
            tone = tone_marks[format(ord(letter.lower()),'04x')]
            main_vowel = normalized[idx-1]
            main_vowel_idx = idx-1
    if tone:
        syllable = normalized[:main_vowel_idx]+main_vowel+normalized[main_vowel_idx+2:]
    else:
        if syllable.endswith(PingImSyllable.CHECKED_TONES):
            tone = '4'
        else:
            tone = '1'

    result = syllable+tone
    result = unicodedata.normalize("NFC",result)

    return result

class TaigiConverter:
    def __init__(self) -> None:
        self.mode: int = None
        self.text: str = None
    def update(self, text: str, mode: int) -> "TaigiConverter":
        self.text: str = text
        self.mode: int = mode
        return self
    

    @staticmethod
    def _convert_between_tl_and_poj(text: str) -> str:
        # 請實作
        return text
    
    @staticmethod
    def _convert_between_no_and_diacritics(text: str, mode: int) -> str:
        # 請實作
        pattern = re.compile(r'[a-z0-9A-Z\u0301\u0300\u0302\u030c\u0304\u030d\u030b\-]+')
        target = unicodedata.normalize("NFD",text)
        matches = pattern.findall(target)
        matches = [item for sublist in matches for item in sublist.split('-')]
        for match in sorted(matches, key=len,reverse=True):
            if mode == 2 :
                added = process_pinyin(match[:-1],match[-1:])
                target = target.replace(match,added)
            else:
                removal = diacritic_removal(match)
                target = target.replace(match, removal)
        text = target
            
        return text

    def convert(self) -> str:
        if 0 <= self.mode < 2:
            return self._convert_between_tl_and_poj(self.text,self.mode)
        if 2 <= self.mode < 4:
            return self._convert_between_no_and_diacritics(self.text, self.mode)
        raise ValueError("Unsupported Mode!")

In [166]:
# input_text: str = "guá sī ông-iok-tik"
taigi_converter = TaigiConverter()
# taigi_converter.update(TESTSTR_DIAC2NO, TaigiModes.MODE_DIAC2NO)
taigi_converter.update(TESTSTR_NO2DIAC, TaigiModes.MODE_NO2DIAC)
output_text: str = taigi_converter.convert()
print(output_text)
# assert(output_text == "gua2 si7 ong5-iok4-tik4")

Ji̍p-tang í-āu, m̄-nā thinn-khì tsuán-líng, liû-hîng-sìng kám-mōō ê pēnn-to̍k mā tuè-leh khai-sí ua̍h-thiàu, tsō-sîng put-tsí-á tsē ha̍k-sing gín-á kap tuā-lâng tio̍h-pēnn


### Test for parse long string

In [54]:
word = 'ǎa̍a̋'

tone_marks:dict = {'0301':'2','0300':'3','0302':'5','030c':'6','0304':'7','030d':'8','030b':'9',
                   '2':'0301','3':'0300','5':'0302','6':'030c','7':'0304','8':'030d','9':'030b'}
vowel_order = ['a','e','i','oo','o','u']



for letter in word:
    decomposed = unicodedata.decomposition(letter)
    print(f"{letter}: {hex(ord(letter))} = {decomposed}")

# print(decomposed)

ǎ: 0x1ce = 0061 030C
a: 0x61 = 
̍: 0x30d = 
a: 0x61 = 
̋: 0x30b = 


In [55]:
word = 'a̍'
word = 'â'
# decomposed = unicodedata.decomposition(word).split()
# tone_marks[decomposed.split()[1]]
# b=chr(int(decomposed[0],16))
# b
n = unicodedata.normalize('NFD',word)

for t in list(n):
    print(f"{t}  :  {unicodedata.category(t)}")

print(len(word),len(n))

a  :  Ll
̂  :  Mn
1 2


In [64]:
normalized = unicodedata.normalize('NFD',TESTSTR_DIAC2NO)
words = normalized.split(' ')

# a = 'asdasdasdasd'
# a.split('-')

for word in words:
    characters = word.split('-')
    if len(characters) < 2 : #check if the word is single-characters
        unicodedata.decomposition

In [55]:
def __diacritic_removal(syllable:str):
    normalized = unicodedata.normalize('NFD',syllable)
    tone=''
    for idx, letter in enumerate(normalized):
        if unicodedata.category(letter) == 'Mn' and letter != '\u0358' :
            tone = tone_marks[format(ord(letter.lower()),'04x')]
            main_vowel = normalized[idx-1]
            main_vowel_idx = idx-1
    if tone:
        syllable = normalized[:main_vowel_idx]+main_vowel+normalized[main_vowel_idx+2:]
    return syllable+tone

In [165]:
import re

pattern = re.compile(r'[a-z0-9A-Z\u0301\u0300\u0302\u030c\u0304\u030d\u030b\-]+')
target = unicodedata.normalize("NFD",TESTSTR_NO2DIAC)
matches = pattern.findall(target)
matches = [item for sublist in matches for item in sublist.split('-')]

for match in sorted(matches, key=len,reverse=True):
    # removal = diacritic_removal(match)
    print(f"{match} -> {match[:-1]}: {match[-1:]}")
    processed = process_pinyin(match[:-1],match[-1:])
    # target = target.replace(match, removal)
    # target = target.replace(match, processed)

# target

thinn1 -> thinn: 1
tsuan2 -> tsuan: 2
thiau3 -> thiau: 3
tang1 -> tang: 1
ling2 -> ling: 2
hing5 -> hing: 5
sing3 -> sing: 3
penn7 -> penn: 7
khai1 -> khai: 1
sing5 -> sing: 5
sing1 -> sing: 1
lang5 -> lang: 5
tioh8 -> tioh: 8
penn7 -> penn: 7
Jip8 -> Jip: 8
khi3 -> khi: 3
liu5 -> liu: 5
kam2 -> kam: 2
moo7 -> moo: 7
tok8 -> tok: 8
tue3 -> tue: 3
leh4 -> leh: 4
uah8 -> uah: 8
tso7 -> tso: 7
put4 -> put: 4
tsi2 -> tsi: 2
tse7 -> tse: 7
hak8 -> hak: 8
gin2 -> gin: 2
kap4 -> kap: 4
tua7 -> tua: 7
au7 -> au: 7
na7 -> na: 7
ma7 -> ma: 7
si2 -> si: 2
i2 -> i: 2
m7 -> m: 7
e5 -> e: 5
a2 -> a: 2
a2 -> a: 2


In [39]:

# 使用示例
if __name__ == "__main__":
    test_cases = [
        ("ma", "2"),
        ("hui", "2"),
        ("tiong", "5"),
        ("ing", "3"),
        ('kui','3'),
        ('hiu','2'),
        ('hng','7'),
        ('hainnh','8'),
        ('hmh','8')
    ]
    
    for syllable, tone in test_cases:
        try:
            result = process_pinyin(syllable, tone)
            print(f"{syllable} + {tone} 聲調 -> {result}")
        except ValueError as e:
            print(f"處理 {syllable} 時發生錯誤：{str(e)}")

ma + 2 聲調 -> má
hui + 2 聲調 -> huí
tiong + 5 聲調 -> tiông
ing + 3 聲調 -> ìng
kui + 3 聲調 -> kuì
hiu + 2 聲調 -> hiú
hng + 7 聲調 -> hn̄g
hainnh + 8 聲調 -> ha̍innh
處理 hmh 時發生錯誤：母音數量必須在1-3個之間，目前有0個


In [204]:
#測試可行的新轉換parsing

import re

_input = TESTSTR_NO2DIAC

pattern = re.compile(r'[a-z0-9A-Z\u0301\u0300\u0302\u030c\u0304\u030d\u030b\-]+')
target = unicodedata.normalize("NFD",_input)

matches = pattern.findall(target)
matches = [item for sublist in matches for item in sublist.split('-')]

start=0
end=0

new = ''

for match in matches:
    # print(target.find(match))
    end = target.find(match,start)
    # print(f"\nfillings between {start} to {end} : {target[start:end]}\n")
    new += target[start:end]
    # print(f"{match}: {target.find(match,start)} -> {match[:-1]}:{match[-1:]}")
    processed = process_pinyin(match[:-1],match[-1:])
    # print(processed)
    new += processed
    start = end+len(match)

new+=target[start:]

print(new)




Ji̍p-tang í-āu, m̄-nā thinn-khì tsuán-líng, liû-hîng-sìng kám-mōō ê pēnn-to̍k mā tuè-leh khai-sí ua̍h-thiàu, tsō-sîng put-tsí-á tsē ha̍k-sing gín-á kap tuā-lâng tio̍h-pēnn.


In [206]:
a= process_pinyin("moo",'7')
a

'mōō'